## Exploratory Data Analysis

### Import Libraries

In [4]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [5]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [6]:
df = pl.read_csv(
    "../couchbase_scripts/data/Household.csv", 
    ignore_errors=True, 
    truncate_ragged_lines=True
    )

df

CUST_ID,HOUSEHOLD_ID,ADDRESS,CITY,COUNTRY,STATE,ZIP,ADDRESS_LAST_CHANGED_DATE,NUMBER_OF_DEPENDENT_ADULTS,NUMBER_OF_DEPENDENT_CHILDREN,FAMILY_SIZE,HEAD_OF_HOUSEHOLD_INDICATOR,HOME_OWNER_INDICATOR,URBAN_CODE,PRIMARY_ADVISOR_ID
str,str,str,str,str,str,str,str,i64,i64,i64,bool,bool,str,i64
"""CUST-417911""","""HHID-417911""","""3629 Alpine Trail""","""Humble""","""United States""","""Texas""","""77346""","""5/4/15""",0,1,3,true,true,"""City""",50662
"""CUST-758898""","""HHID-758898""","""69857 Winsor St""","""Sacramento""","""USA""","""CA""","""95827""","""5/4/15""",0,1,3,true,true,"""City""",53281
"""CUST-958684""","""HHID-958684""","""4702 Dublin Blvd""","""Denver""","""USA""","""CO""","""80216""","""6/12/15""",0,1,3,true,true,"""City""",108919
"""CUST-124574""","""HHID-124574""","""6 Riverdale Rd #27""","""Monroeville""","""USA""","""PA""","""15146""","""5/13/16""",0,0,2,true,false,"""Urban""",104140
"""CUST-198781""","""HHID-198781""","""0 La Follette Center""","""Denver""","""United States""","""Colorado""","""80241""","""5/13/16""",0,0,2,true,false,"""Urban""",139823
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CUST-717296""","""HHID-717296""","""6 Bunting Parkway""","""Stockton""","""United States""","""California""","""95205""","""1/31/15""",0,0,2,true,false,"""Urban""",57742
"""CUST-780694""","""HHID-780694""","""3 Mcauley Dr""","""Ashland""","""USA""","""OH""","""44805""","""1/31/15""",0,0,2,true,false,"""Urban""",93736
"""CUST-787420""","""HHID-787420""","""639 Main St""","""Anchorage""","""USA""","""AK""","""99501""","""3/27/15""",0,0,1,true,false,"""Urban""",139141


### Retrieve Number of Nulls in Each Feature

In [7]:
def count_nulls(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "feature": df.columns,
        "null_count": df.null_count().row(0)
    })

pl.Config.set_tbl_rows(35)

null_counts = count_nulls(df)
null_counts

feature,null_count
str,i64
"""CUST_ID""",0
"""HOUSEHOLD_ID""",0
"""ADDRESS""",0
"""CITY""",0
"""COUNTRY""",0
"""STATE""",0
"""ZIP""",0
"""ADDRESS_LAST_CHANGED_DATE""",0
"""NUMBER_OF_DEPENDENT_ADULTS""",0


### Retrieve Basic Information About DataFrame

In [8]:
def print_schema(df: pl.DataFrame):
    print(f"{'Column':<30} | {'Data Type'}")
    print("-" * 60)
    for name, dtype in zip(df.columns, df.dtypes):
        print(f"{name:<30} | {dtype}")

print_schema(df)

Column                         | Data Type
------------------------------------------------------------
CUST_ID                        | String
HOUSEHOLD_ID                   | String
ADDRESS                        | String
CITY                           | String
COUNTRY                        | String
STATE                          | String
ZIP                            | String
ADDRESS_LAST_CHANGED_DATE      | String
NUMBER_OF_DEPENDENT_ADULTS     | Int64
NUMBER_OF_DEPENDENT_CHILDREN   | Int64
FAMILY_SIZE                    | Int64
HEAD_OF_HOUSEHOLD_INDICATOR    | Boolean
HOME_OWNER_INDICATOR           | Boolean
URBAN_CODE                     | String
PRIMARY_ADVISOR_ID             | Int64


### Display Summary Statistics for All Columns

In [9]:
summary = df.describe()
print(summary)

shape: (9, 16)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ CUST_ID   ┆ HOUSEHOLD ┆ ADDRESS   ┆ … ┆ HEAD_OF_H ┆ HOME_OWNE ┆ URBAN_COD ┆ PRIMARY_ │
│ ---       ┆ ---       ┆ _ID       ┆ ---       ┆   ┆ OUSEHOLD_ ┆ R_INDICAT ┆ E         ┆ ADVISOR_ │
│ str       ┆ str       ┆ ---       ┆ str       ┆   ┆ INDICATOR ┆ OR        ┆ ---       ┆ ID       │
│           ┆           ┆ str       ┆           ┆   ┆ ---       ┆ ---       ┆ str       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 2000      ┆ 2000      ┆ 2000      ┆ … ┆ 2000.0    ┆ 2000.0    ┆ 2000      ┆ 2000.0   │
│ null_coun ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0.0       ┆ 0.0       ┆ 0         ┆ 0.0      │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           

### Find Longest Text Length in Each Column

In [10]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

CUST_ID,HOUSEHOLD_ID,ADDRESS,CITY,COUNTRY,STATE,ZIP,ADDRESS_LAST_CHANGED_DATE,URBAN_CODE
u32,u32,u32,u32,u32,u32,u32,u32,u32
11,11,31,26,13,25,5,8,5


### Retrieve Data Types of All Columns

In [11]:
print("Column data types:\n", df.dtypes)

Column data types:
 [String, String, String, String, String, String, String, String, Int64, Int64, Int64, Boolean, Boolean, String, Int64]


### Count Unique Values in Each Column

In [12]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                      Unique values in CUST_ID : 1999  
                 Unique values in HOUSEHOLD_ID : 1999  
                      Unique values in ADDRESS : 2000  
                         Unique values in CITY : 885   
                      Unique values in COUNTRY : 3     
                        Unique values in STATE : 110   
                          Unique values in ZIP : 1475  
    Unique values in ADDRESS_LAST_CHANGED_DATE : 688   
   Unique values in NUMBER_OF_DEPENDENT_ADULTS : 1     
 Unique values in NUMBER_OF_DEPENDENT_CHILDREN : 5     
                  Unique values in FAMILY_SIZE : 4     
  Unique values in HEAD_OF_HOUSEHOLD_INDICATOR : 2     
         Unique values in HOME_OWNER_INDICATOR : 2     
                   Unique values in URBAN_CODE : 3     
           Unique values in PRIMARY_ADVISOR_ID : 995   


### Check Distribution of Numerical Columns

In [13]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['ID']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

NUMBER_OF_DEPENDENT_ADULTS
shape: (9, 2)
┌────────────┬────────────────────────────┐
│ statistic  ┆ NUMBER_OF_DEPENDENT_ADULTS │
│ ---        ┆ ---                        │
│ str        ┆ f64                        │
╞════════════╪════════════════════════════╡
│ count      ┆ 2000.0                     │
│ null_count ┆ 0.0                        │
│ mean       ┆ 0.0                        │
│ std        ┆ 0.0                        │
│ min        ┆ 0.0                        │
│ 25%        ┆ 0.0                        │
│ 50%        ┆ 0.0                        │
│ 75%        ┆ 0.0                        │
│ max        ┆ 0.0                        │
└────────────┴────────────────────────────┘ 


NUMBER_OF_DEPENDENT_CHILDREN
shape: (9, 2)
┌────────────┬──────────────────────────────┐
│ statistic  ┆ NUMBER_OF_DEPENDENT_CHILDREN │
│ ---        ┆ ---                          │
│ str        ┆ f64                          │
╞════════════╪══════════════════════════════╡
│ count      ┆ 2000.0  

### List Unique Values For Certain Features

In [14]:
def list_unique_values_under_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count < threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_under_threshold(df)

Column: CUST_ID (1999 unique values)
shape: (1_999,)
Series: 'CUST_ID' [str]
[
	"CUST-100067"
	"CUST-100282"
	"CUST-100396"
	"CUST-100400"
	"CUST-100417"
	"CUST-100655"
	"CUST-100759"
	"CUST-101785"
	"CUST-101865"
	"CUST-102995"
	"CUST-103026"
	"CUST-103108"
	"CUST-103227"
	"CUST-104309"
	"CUST-104713"
	"CUST-105166"
	"CUST-106996"
	"CUST-107207"
	…
	"CUST-993400"
	"CUST-994052"
	"CUST-994211"
	"CUST-994613"
	"CUST-994795"
	"CUST-994918"
	"CUST-994965"
	"CUST-995095"
	"CUST-996204"
	"CUST-997038"
	"CUST-997072"
	"CUST-997260"
	"CUST-997681"
	"CUST-998954"
	"CUST-998988"
	"CUST-999095"
	"CUST-999182"
]
--------------------------------------------------
Column: HOUSEHOLD_ID (1999 unique values)
shape: (1_999,)
Series: 'HOUSEHOLD_ID' [str]
[
	"HHID-100067"
	"HHID-100282"
	"HHID-100396"
	"HHID-100400"
	"HHID-100417"
	"HHID-100655"
	"HHID-100759"
	"HHID-101785"
	"HHID-101865"
	"HHID-102995"
	"HHID-103026"
	"HHID-103108"
	"HHID-103227"
	"HHID-104309"
	"HHID-104713"
	"HHID-105166"
	"HHID-1069

In [15]:
def list_unique_values_over_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count > threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_over_threshold(df)

### How Many Records Remain IF I Remove Records With Any Nulls In It

In [16]:
def drop_rows_with_any_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """
    Removes all rows from a Polars DataFrame that contain any null values.
    """
    return df.drop_nulls()


drop_rows_with_any_nulls(df)

CUST_ID,HOUSEHOLD_ID,ADDRESS,CITY,COUNTRY,STATE,ZIP,ADDRESS_LAST_CHANGED_DATE,NUMBER_OF_DEPENDENT_ADULTS,NUMBER_OF_DEPENDENT_CHILDREN,FAMILY_SIZE,HEAD_OF_HOUSEHOLD_INDICATOR,HOME_OWNER_INDICATOR,URBAN_CODE,PRIMARY_ADVISOR_ID
str,str,str,str,str,str,str,str,i64,i64,i64,bool,bool,str,i64
"""CUST-417911""","""HHID-417911""","""3629 Alpine Trail""","""Humble""","""United States""","""Texas""","""77346""","""5/4/15""",0,1,3,true,true,"""City""",50662
"""CUST-758898""","""HHID-758898""","""69857 Winsor St""","""Sacramento""","""USA""","""CA""","""95827""","""5/4/15""",0,1,3,true,true,"""City""",53281
"""CUST-958684""","""HHID-958684""","""4702 Dublin Blvd""","""Denver""","""USA""","""CO""","""80216""","""6/12/15""",0,1,3,true,true,"""City""",108919
"""CUST-124574""","""HHID-124574""","""6 Riverdale Rd #27""","""Monroeville""","""USA""","""PA""","""15146""","""5/13/16""",0,0,2,true,false,"""Urban""",104140
"""CUST-198781""","""HHID-198781""","""0 La Follette Center""","""Denver""","""United States""","""Colorado""","""80241""","""5/13/16""",0,0,2,true,false,"""Urban""",139823
"""CUST-228676""","""HHID-228676""","""410 Acco Dr""","""Glendale""","""USA""","""CA""","""91203""","""1/8/15""",0,0,1,true,false,"""Urban""",142557
"""CUST-464124""","""HHID-464124""","""192 Otis St""","""Anderson""","""USA""","""IN""","""46013""","""8/30/15""",0,0,1,true,true,"""City""",74963
"""CUST-529407""","""HHID-529407""","""6768 Heath Pass""","""Scranton""","""United States""","""Pennsylvania""","""18505""","""8/30/15""",0,0,1,true,true,"""City""",51943
"""CUST-523002""","""HHID-523002""","""981 Hanson Park""","""Naples""","""United States""","""Florida""","""34108""","""4/9/15""",0,1,3,true,true,"""City""",47954
